In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step8 Final Prognostic Model
# Step8.1 Initialize
# ============================================================

import os
import pandas as pd
import numpy as np


project_dir = _hcc_path('')


step8_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step8_Final_Model"
)


result_dir = os.path.join(
    step8_dir,
    "Results"
)


fig_dir = os.path.join(
    step8_dir,
    "Figures"
)


model_dir = os.path.join(
    step8_dir,
    "Models"
)


for d in [
    step8_dir,
    result_dir,
    fig_dir,
    model_dir
]:
    os.makedirs(d, exist_ok=True)


print(step8_dir)

In [ ]:
# ============================================================
# Step8.2 RNA
# ============================================================


rna_file = os.path.join(
    project_dir,
    "LIHC_RNA_Step1",
    "LIHC_RNA_Normalized_371.tsv"
)


rna = pd.read_csv(
    rna_file,
    sep="\t",
    index_col=0
)


print(
    "RNA shape:",
    rna.shape
)


display(
    rna.iloc[:5,:5]
)

In [ ]:
# ============================================================
# Step8.3 Survival
# ============================================================


clinical_file = os.path.join(
    project_dir,
    "LIHC_Clinical_Survival_Final_371.csv"
)


clinical = pd.read_csv(
    clinical_file
)


print(
    clinical.shape
)


display(
    clinical[
        [
            "Sample_ID",
            "OS_time",
            "OS"
        ]
    ].head()
)

In [ ]:
# ============================================================
# Step8.4 Filter survival
# ============================================================


clinical_surv = clinical.dropna(
    subset=[
        "OS_time",
        "OS"
    ]
).copy()


clinical_surv = clinical_surv.set_index(
    "Sample_ID"
)


print(
    "Survival patients:",
    clinical_surv.shape[0]
)


print(
    clinical_surv["OS"].value_counts()
)

In [ ]:
# ============================================================
# Step8.5 Match
# ============================================================


X = rna.T.copy()


common_samples = (
    clinical_surv.index
    .intersection(
        X.index
    )
)


X = X.loc[
    common_samples
]


clinical_surv = clinical_surv.loc[
    common_samples
]


print(
    "Expression:",
    X.shape
)


print(
    "Clinical:",
    clinical_surv.shape
)


print(
    "Order:",
    X.index.equals(
        clinical_surv.index
    )
)

In [ ]:
# ============================================================
# Step8.6 Survival object
# ============================================================


from sksurv.util import Surv


y_surv = Surv.from_arrays(
    event=
    clinical_surv["OS"]
    .astype(bool)
    .values,

    time=
    clinical_surv["OS_time"]
    .astype(float)
    .values
)


print(
    y_surv[:5]
)

In [ ]:
# ============================================================
# Step8.7 Univariate Cox
# ============================================================


from lifelines import CoxPHFitter


cox_results = []


cph = CoxPHFitter()


for i, gene in enumerate(X.columns):

    temp = pd.DataFrame({

        "OS_time":
        clinical_surv["OS_time"],

        "OS":
        clinical_surv["OS"],

        "Gene":
        X[gene]

    })


    if temp["Gene"].std()==0:
        continue


    try:

        cph.fit(
            temp,
            duration_col="OS_time",
            event_col="OS"
        )


        cox_results.append([

            gene,

            np.exp(
                cph.params_["Gene"]
            ),

            cph.summary.loc[
                "Gene",
                "p"
            ]

        ])

    except:
        pass


    if (i+1)%1000==0:
        print(
            i+1,
            "/",
            X.shape[1]
        )


print("Finished")

In [ ]:
cox_df = pd.DataFrame(
    cox_results,
    columns=[
        "Gene",
        "HR",
        "P_value"
    ]
)


cox_df = cox_df.sort_values(
    "P_value"
)


print(
    cox_df.shape
)


display(
    cox_df.head(20)
)


cox_df.to_csv(
    os.path.join(
        result_dir,
        "Univariate_Cox_AllGenes.csv"
    ),
    index=False
)

In [ ]:
from statsmodels.stats.multitest import multipletests


cox_df["FDR"] = multipletests(
    cox_df["P_value"],
    method="fdr_bh"
)[1]


cox_sig = cox_df[
    cox_df["FDR"]<0.05
].copy()


print(
    "FDR<0.05 genes:",
    cox_sig.shape[0]
)


cox_sig.to_csv(
    os.path.join(
        result_dir,
        "Univariate_Cox_FDR005.csv"
    ),
    index=False
)

In [ ]:
print(cox_df.shape)

print(
"FDR<0.05:",
cox_sig.shape[0]
)

display(cox_sig.head(20))

In [ ]:
# ============================================================
# Step8.10
# Prepare LASSO genes
# ============================================================


candidate_genes = cox_sig["Gene"].tolist()


print(
    "Candidate genes:",
    len(candidate_genes)
)


X_lasso = X[
    candidate_genes
].copy()


print(
    X_lasso.shape
)

In [ ]:
# ============================================================
# Standardization
# ============================================================


from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()


X_lasso_scaled = scaler.fit_transform(
    X_lasso
)


X_lasso_scaled = pd.DataFrame(
    X_lasso_scaled,
    index=X_lasso.index,
    columns=X_lasso.columns
)


print(
    X_lasso_scaled.shape
)

In [ ]:
# ============================================================
# LASSO Cox path
# ============================================================


from sksurv.linear_model import CoxnetSurvivalAnalysis


coxnet = CoxnetSurvivalAnalysis(
    l1_ratio=1.0,
    n_alphas=100,
    alpha_min_ratio=0.01,
    max_iter=100000
)


coxnet.fit(
    X_lasso_scaled,
    y_surv
)


alphas = coxnet.alphas_


print(
    "Number of alpha:",
    len(alphas)
)

In [ ]:
# ============================================================
# Non-zero genes
# ============================================================


coef_path = coxnet.coef_


nonzero_counts = (
    np.abs(coef_path)>1e-8
).sum(axis=0)


alpha_table = pd.DataFrame({

    "alpha":alphas,

    "nonzero_genes":
    nonzero_counts

})


display(
    alpha_table.head(20)
)

In [ ]:
print("Alpha number:",len(alphas))

print(
alpha_table.tail()
)

print(
"Min genes:",
nonzero_counts.min()
)

print(
"Max genes:",
nonzero_counts.max()
)

In [ ]:
# ============================================================
# Step8.11
# 5-fold CV for LASSO-Cox
# ============================================================


from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_censored


event = clinical_surv["OS"].astype(int)


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)


cv_scores = np.zeros(
    (len(alphas),5)
)



for fold,(train_idx,val_idx) in enumerate(
    skf.split(
        X_lasso_scaled,
        event
    )
):

    print(
        f"Fold {fold+1}/5"
    )


    X_train_fold = (
        X_lasso_scaled.iloc[train_idx]
    )


    X_val_fold = (
        X_lasso_scaled.iloc[val_idx]
    )


    y_train_fold = (
        y_surv[train_idx]
    )


    y_val_fold = (
        y_surv[val_idx]
    )


    model = CoxnetSurvivalAnalysis(
        l1_ratio=1.0,
        alphas=alphas,
        max_iter=100000
    )


    model.fit(
        X_train_fold,
        y_train_fold
    )


    for i,alpha in enumerate(alphas):

        try:

            risk = model.predict(
                X_val_fold,
                alpha=alpha
            )


            cindex = concordance_index_censored(
                y_val_fold["event"],
                y_val_fold["time"],
                risk
            )[0]


            cv_scores[i,fold]=cindex


        except:

            cv_scores[i,fold]=np.nan



print("CV finished")

In [ ]:
# ============================================================
# CV summary
# ============================================================


cv_mean = np.nanmean(
    cv_scores,
    axis=1
)


cv_std = np.nanstd(
    cv_scores,
    axis=1
)


cv_result = pd.DataFrame({

    "alpha":alphas,

    "CV_Cindex":
    cv_mean,

    "SD":
    cv_std,

    "Genes":
    nonzero_counts

})


display(
    cv_result.sort_values(
        "CV_Cindex",
        ascending=False
    ).head(20)
)

In [ ]:
best_idx = np.nanargmax(
    cv_mean
)


best_alpha = alphas[
    best_idx
]


print(
    "Best alpha:",
    best_alpha
)


print(
    "Best CV C-index:",
    cv_mean[best_idx]
)


print(
    "Genes:",
    nonzero_counts[best_idx]
)

In [ ]:
# ============================================================
# Step8.12
# Extract LASSO coefficients
# ============================================================


best_coef = pd.DataFrame({

    "Gene":
    X_lasso.columns,

    "Coefficient":
    coxnet.coef_[:,best_idx]

})


best_coef = best_coef[
    best_coef["Coefficient"]!=0
]


best_coef["Abs_Coefficient"] = (
    best_coef["Coefficient"]
    .abs()
)


best_coef = best_coef.sort_values(
    "Abs_Coefficient",
    ascending=False
)


print(
    best_coef.shape
)


display(
    best_coef.head(30)
)

In [ ]:
# ============================================================
# Step8.13
# Top coefficient genes
# ============================================================

top10_genes = best_coef.head(10)["Gene"].tolist()

top10_genes

In [ ]:
# ============================================================
# Step8.13.1
# Top10 gene multivariate Cox
# ============================================================


from lifelines import CoxPHFitter


top10_genes = best_coef.head(10)["Gene"].tolist()


cox_top10_data = X[top10_genes].copy()


cox_top10_data["OS_time"] = (
    clinical_surv["OS_time"]
)


cox_top10_data["OS"] = (
    clinical_surv["OS"]
)


cph_top10 = CoxPHFitter()


cph_top10.fit(
    cox_top10_data,
    duration_col="OS_time",
    event_col="OS"
)


display(
    cph_top10.summary
)

In [ ]:
# ============================================================
# RiskScore
# ============================================================


coef_top10 = cph_top10.params_


risk_top10 = (
    X[top10_genes]
    .dot(coef_top10)
)


risk_top10.head()

In [ ]:
# ============================================================
# Risk group
# ============================================================


cutoff = risk_top10.median()


group_top10 = pd.DataFrame({

    "RiskScore":
    risk_top10,

    "RiskGroup":
    np.where(
        risk_top10 > cutoff,
        "High",
        "Low"
    )

})


display(
    group_top10.head()
)


print(
    group_top10["RiskGroup"].value_counts()
)

In [ ]:
# ============================================================
# KM
# ============================================================


from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt


kmf = KaplanMeierFitter()


fig, ax = plt.subplots(
    figsize=(7,6)
)


for g in ["High","Low"]:

    idx = (
        group_top10["RiskGroup"]
        ==
        g
    )


    kmf.fit(
        clinical_surv.loc[
            idx,
            "OS_time"
        ],

        clinical_surv.loc[
            idx,
            "OS"
        ],

        label=g
    )


    kmf.plot_survival_function(
        ax=ax
    )


high = group_top10[
    group_top10["RiskGroup"]=="High"
].index


low = group_top10[
    group_top10["RiskGroup"]=="Low"
].index



result = logrank_test(

    clinical_surv.loc[
        high,
        "OS_time"
    ],

    clinical_surv.loc[
        low,
        "OS_time"
    ],

    event_observed_A=
    clinical_surv.loc[
        high,
        "OS"
    ],

    event_observed_B=
    clinical_surv.loc[
        low,
        "OS"
    ]

)


plt.title(
    f"Top10 signature KM\np={result.p_value:.4g}"
)


plt.show()


print(
    "Log-rank p:",
    result.p_value
)

In [ ]:
from lifelines.utils import concordance_index


cindex_top10 = concordance_index(

    clinical_surv["OS_time"],

    -risk_top10,

    clinical_surv["OS"]

)


print(
    "Top10 C-index:",
    cindex_top10
)

In [ ]:
# ============================================================
# Step8.14.1
# Bootstrap validation preparation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.utils import resample
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index


# 最终10基因

final_genes = top10_genes


print(
    "Final genes:"
)

print(final_genes)

In [ ]:
# ============================================================
# Step8.14.2
# Bootstrap C-index
# ============================================================


n_bootstrap = 1000


bootstrap_cindex = []


data_boot = X[final_genes].copy()


data_boot["OS_time"] = (
    clinical_surv["OS_time"]
)


data_boot["OS"] = (
    clinical_surv["OS"]
)



for i in range(n_bootstrap):

    if (i+1)%100 == 0:
        print(
            f"Bootstrap {i+1}/{n_bootstrap}"
        )


    # -------------------------
    # 有放回抽样
    # -------------------------

    boot_data = resample(
        data_boot,
        replace=True,
        random_state=2026+i
    )


    try:

        cph = CoxPHFitter()


        cph.fit(
            boot_data,
            duration_col="OS_time",
            event_col="OS"
        )


        # 预测风险

        risk = cph.predict_partial_hazard(
            boot_data[final_genes]
        )


        cindex = concordance_index(

            boot_data["OS_time"],

            -risk.values,

            boot_data["OS"]

        )


        bootstrap_cindex.append(
            cindex
        )


    except:

        continue



print("Bootstrap finished")

In [ ]:
# ============================================================
# Step8.14.3
# Summary
# ============================================================


bootstrap_cindex = np.array(
    bootstrap_cindex
)


mean_cindex = np.mean(
    bootstrap_cindex
)


lower = np.percentile(
    bootstrap_cindex,
    2.5
)


upper = np.percentile(
    bootstrap_cindex,
    97.5
)


print(
    "Bootstrap samples:",
    len(bootstrap_cindex)
)


print(
    f"Mean C-index: {mean_cindex:.4f}"
)


print(
    f"95% CI: {lower:.4f}-{upper:.4f}"
)

In [ ]:
# ============================================================
# Save
# ============================================================


bootstrap_result = pd.DataFrame({

    "C_index":
    bootstrap_cindex

})


bootstrap_result.to_csv(
    os.path.join(
        result_dir,
        "Top10_Bootstrap_Cindex.csv"
    ),
    index=False
)


print("Saved")

In [ ]:
# ============================================================
# Step8.15.1
# Final Risk Score
# ============================================================

final_cph = CoxPHFitter()


final_data = X[final_genes].copy()

final_data["OS_time"] = (
    clinical_surv["OS_time"]
)

final_data["OS"] = (
    clinical_surv["OS"]
)


final_cph.fit(
    final_data,
    duration_col="OS_time",
    event_col="OS"
)


final_coef = final_cph.params_


display(
    final_coef
)

In [ ]:
# RiskScore

risk_score_final = (
    X[final_genes]
    .dot(final_coef)
)


risk_score_final.head()

In [ ]:
risk_table = pd.DataFrame({

    "Sample_ID":
    risk_score_final.index,

    "RiskScore":
    risk_score_final.values,

    "OS_time":
    clinical_surv["OS_time"],

    "OS":
    clinical_surv["OS"]

})


risk_table.to_csv(
    os.path.join(
        result_dir,
        "Final_Top10_RiskScore.csv"
    ),
    index=False
)


display(
    risk_table.head()
)

In [ ]:
# ============================================================
# Step8.16 Final version
# Risk score distribution + Survival status
# ============================================================


import matplotlib.pyplot as plt
import numpy as np
import os


# ==============================
# Prepare data
# ==============================

risk_plot = risk_table.copy()


# sort by risk score

risk_plot = (
    risk_plot
    .sort_values(
        "RiskScore"
    )
    .reset_index(drop=True)
)


risk_plot["Index"] = np.arange(
    risk_plot.shape[0]
)


# median cutoff

cutoff = risk_plot["RiskScore"].median()


risk_plot["RiskGroup"] = np.where(
    risk_plot["RiskScore"] > cutoff,
    "High risk",
    "Low risk"
)


# ==============================
# Figure
# ==============================


fig, axes = plt.subplots(
    2,
    1,
    figsize=(10,8),
    sharex=True,
    gridspec_kw={
        "height_ratios":[2,1]
    }
)


# ==================================================
# Panel A Risk score
# ==================================================


low = risk_plot[
    risk_plot["RiskGroup"]=="Low risk"
]


high = risk_plot[
    risk_plot["RiskGroup"]=="High risk"
]


axes[0].scatter(
    low["Index"],
    low["RiskScore"],
    s=18,
    label="Low risk"
)


axes[0].scatter(
    high["Index"],
    high["RiskScore"],
    s=18,
    label="High risk"
)


axes[0].axhline(
    cutoff,
    linestyle="--"
)


axes[0].set_ylabel(
    "Risk score"
)


axes[0].set_title(
    "Risk score distribution"
)


axes[0].legend(
    frameon=False
)



# ==================================================
# Panel B Survival status
# ==================================================


alive = risk_plot[
    risk_plot["OS"]==0
]


death = risk_plot[
    risk_plot["OS"]==1
]


axes[1].scatter(
    alive["Index"],
    alive["OS"],
    s=18,
    label="Alive"
)


axes[1].scatter(
    death["Index"],
    death["OS"],
    s=18,
    label="Dead"
)


axes[1].axvline(
    len(low),
    linestyle="--"
)


axes[1].set_xlabel(
    "Patients ranked by risk score"
)


axes[1].set_ylabel(
    "Survival status"
)


axes[1].set_yticks(
    [0,1]
)


axes[1].legend(
    frameon=False
)



# ==============================
# Save
# ==============================


plt.tight_layout()


save_path = os.path.join(
    fig_dir,
    "Figure_RiskScore_SurvivalStatus_Final.png"
)


plt.savefig(
    save_path,
    dpi=600,
    bbox_inches="tight"
)


plt.show()


plt.close()


print(
    "Saved:",
    save_path
)

In [ ]:
# ============================================================
# Step8.17 KM curve final publication version
# ============================================================


import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.plotting import add_at_risk_counts
import numpy as np
import pandas as pd
import os



# ============================================================
# Prepare data
# ============================================================

km_data = risk_table.copy()


cutoff = km_data["RiskScore"].median()


km_data["RiskGroup"] = np.where(
    km_data["RiskScore"] > cutoff,
    "High risk",
    "Low risk"
)



high = km_data[
    km_data["RiskGroup"]=="High risk"
]


low = km_data[
    km_data["RiskGroup"]=="Low risk"
]


print(km_data["RiskGroup"].value_counts())



# ============================================================
# KM fitting
# ============================================================


kmf_high = KaplanMeierFitter()

kmf_low = KaplanMeierFitter()



fig, ax = plt.subplots(
    figsize=(8,7)
)



kmf_low.fit(
    low["OS_time"],
    low["OS"],
    label="Low risk"
)


kmf_high.fit(
    high["OS_time"],
    high["OS"],
    label="High risk"
)



kmf_low.plot_survival_function(
    ax=ax,
    ci_show=True,
    linewidth=2
)


kmf_high.plot_survival_function(
    ax=ax,
    ci_show=True,
    linewidth=2
)



# ============================================================
# Log-rank
# ============================================================


lr = logrank_test(

    high["OS_time"],

    low["OS_time"],

    event_observed_A=high["OS"],

    event_observed_B=low["OS"]

)



# ============================================================
# Cox HR
# ============================================================


cox_df = km_data[
    [
        "OS_time",
        "OS",
        "RiskGroup"
    ]
].copy()


cox_df["RiskGroup"] = np.where(
    cox_df["RiskGroup"]=="High risk",
    1,
    0
)


cox = CoxPHFitter()


cox.fit(
    cox_df,
    duration_col="OS_time",
    event_col="OS"
)



HR = np.exp(
    cox.params_["RiskGroup"]
)


CI = np.exp(
    cox.confidence_intervals_.loc[
        "RiskGroup"
    ]
)


HR_low = CI.iloc[0]

HR_high = CI.iloc[1]



# ============================================================
# Annotation
# ============================================================


text = (
    f"Log-rank p = {lr.p_value:.2e}\n"
    f"HR = {HR:.2f}\n"
    f"95% CI = {HR_low:.2f}-{HR_high:.2f}"
)


ax.text(
    0.58,
    0.25,
    text,
    transform=ax.transAxes,
    fontsize=11
)



# ============================================================
# Add risk table
# ============================================================


add_at_risk_counts(

    kmf_low,
    kmf_high,
    ax=ax

)



# ============================================================
# Formatting
# ============================================================


ax.set_title(
    "Kaplan–Meier survival analysis of the prognostic signature",
    fontsize=14
)


ax.set_xlabel(
    "Time (days)",
    fontsize=12
)


ax.set_ylabel(
    "Overall survival probability",
    fontsize=12
)



ax.legend(
    frameon=False,
    loc="upper right"
)



plt.tight_layout()



# ============================================================
# Save
# ============================================================


png_path = os.path.join(
    fig_dir,
    "KM_Top10_signature_Final.png"
)


svg_path = os.path.join(
    fig_dir,
    "KM_Top10_signature_Final.svg"
)



plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)


plt.savefig(
    svg_path,
    bbox_inches="tight"
)



plt.show()


plt.close()



print("Saved:")
print(png_path)
print(svg_path)


print("\nLog-rank p:",lr.p_value)

print(
    "HR:",
    HR
)

print(
    "95%CI:",
    HR_low,
    "-",
    HR_high
)

In [ ]:
# ============================================================
# Step8.18.1 Time-dependent AUC
# ============================================================


from sksurv.metrics import cumulative_dynamic_auc
from sksurv.util import Surv

import numpy as np



# -----------------------------
# Survival object
# -----------------------------


y_surv = Surv.from_arrays(

    event=
    clinical_surv["OS"].astype(bool),

    time=
    clinical_surv["OS_time"]

)



# -----------------------------
# Risk score
# -----------------------------


risk_score = (
    risk_table["RiskScore"]
    .values
)



# -----------------------------
# Time points
# days
# -----------------------------


eval_times = np.array([

    365,

    1095,

    1825

])



# -----------------------------
# Calculate AUC
# -----------------------------


auc, mean_auc = cumulative_dynamic_auc(

    y_surv,

    y_surv,

    risk_score,

    eval_times

)



for t,a in zip(eval_times,auc):

    print(
        f"{t/365:.0f}-year AUC:",
        round(a,4)
    )


print(
    "Mean AUC:",
    round(mean_auc,4)
)

In [ ]:
# ============================================================
# Step8.18.2 AUC over time
# ============================================================


import matplotlib.pyplot as plt
import os


years = eval_times/365



plt.figure(
    figsize=(7,5)
)



plt.plot(

    years,

    auc,

    marker="o",

    linewidth=2

)



for x,y in zip(years,auc):

    plt.text(

        x,

        y,

        f"{y:.3f}",

        ha="center",

        va="bottom"

    )



plt.ylim(
    0,
    1
)



plt.xlabel(
    "Time (years)"
)



plt.ylabel(
    "AUC"
)



plt.title(
    "Time-dependent ROC analysis"
)



plt.grid(
    alpha=0.3
)



plt.tight_layout()



plt.savefig(

    os.path.join(

        fig_dir,

        "Time_dependent_AUC_Final.png"

    ),

    dpi=600,

    bbox_inches="tight"

)



plt.show()

plt.close()